# Magnol-IA · Probar el MVP en Google Colab

Asistente de voz para que **adultos mayores saquen, consulten, cambien y cancelen citas médicas** hablando como le hablarían a una persona. Grupo 13, Creación de Empresas de Base Tecnológica.

Este cuaderno descarga el proyecto, genera la agenda simulada de **Metrosalud** (sedes, horarios y tiempos de espera reales; pacientes y citas sintéticos) y deja probarlo sin instalar nada. Ejecute las celdas **en orden** con ▶ o `Ctrl + Enter`.

| Paso | Para qué | ¿Obligatorio? |
|---|---|---|
| 1 a 3 | Descargar, configurar y generar la agenda | Sí |
| 4 | Verificar que todo funciona (pruebas automáticas) | Recomendado |
| 5 | Repartir cédulas de prueba entre el equipo | Si prueban varios |
| 6 | Ver una conversación completa de ejemplo | Opcional |
| 7 | Conversar aquí mismo, con respuesta en audio | Opcional |
| 8 | Encender el bot de Telegram para notas de voz | Solo una persona |
| 9 | Métricas y descarga de la bitácora | Al final de la prueba |

## 1. Descargar el proyecto e instalar (≈ 1 minuto)

In [ ]:
import os, subprocess, sys

REPO = "andres0angel/magnol-ia"   # usuario/repositorio en GitHub
CARPETA = "/content/magnolia-mvp"

if os.path.isdir(CARPETA):
    subprocess.run(["git", "-C", CARPETA, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", f"https://github.com/{REPO}.git", CARPETA], check=True)
os.chdir(CARPETA)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("✅ Proyecto listo en", CARPETA)

## 2. Claves (opcionales)

**Sin claves todo funciona con texto**, usando solo reglas. Las claves solo las necesita **la persona que encienda el bot de Telegram** (paso 8):

- `TELEGRAM_TOKEN`: en Telegram busque **@BotFather**, envíe `/newbot` y copie el token.
- `LLM_API_KEY`: clave gratis de Groq (sin tarjeta) en https://console.groq.com/keys. Sirve para entender las **notas de voz** (Whisper) y las frases difíciles (LLM).

Guárdelas en los **Secretos de Colab**: ícono de llave 🔑 en la barra izquierda → *Agregar secreto nuevo* → use exactamente esos nombres y active *Acceso al cuaderno*. No pegue las claves en el texto del cuaderno: si lo comparte, otros las verían.

Si cambia las claves después, vuelva a ejecutar esta celda.

In [ ]:
from pathlib import Path

claves = {}
try:
    from google.colab import userdata
    for nombre in ("TELEGRAM_TOKEN", "LLM_API_KEY"):
        try:
            claves[nombre] = userdata.get(nombre) or ""
        except Exception:
            claves[nombre] = ""
except ImportError:
    pass  # fuera de Colab: use un archivo .env (copie .env.ejemplo)

FECHA_SIMULADA = ""   # p. ej. "2026-11-12T09:00" para una demo con fechas fijas; vacío = la fecha de hoy

lineas = [f"{k}={v}" for k, v in claves.items()]
lineas += [f"FECHA_SIMULADA={FECHA_SIMULADA}", "RESPONDER_VOZ=espejo", "MOSTRAR_TRANSCRIPCION=true"]
Path(".env").write_text("\n".join(lineas) + "\n", encoding="utf-8")

for k in ("TELEGRAM_TOKEN", "LLM_API_KEY"):
    print(f"{k:15} {'✅ cargada' if claves.get(k) else '— no configurada'}")
print("Comprensión:", "reglas + LLM (Groq)" if claves.get("LLM_API_KEY") else "solo reglas (con texto funciona igual)")

## 3. Generar la agenda simulada (≈ 5 segundos)

Crea 51 sedes reales de Metrosalud (dirección, comuna, coordenadas y horario), unos 240 médicos y odontólogos, 1.500 pacientes con la pirámide de edad de Medellín y unos 220.000 cupos calibrados con los días de espera reales de la red. Las fuentes están en `datos/reales/FUENTES.md`.

Si vuelve a ejecutar esta celda, la agenda se crea de nuevo y **las conversaciones anteriores se guardan antes** en `datos/bitacoras/`.

In [ ]:
desde = f"--desde {FECHA_SIMULADA[:10]}" if FECHA_SIMULADA else ""
!python scripts/generar_datos.py {desde}

## 4. Verificar que todo funciona (≈ 10 segundos)

Corre las pruebas automáticas (flujos completos, cancelaciones, urgencias, cuidadores, festivos, datos reales…) y mide cuánto entiende el bot con frases de adultos mayores.

In [ ]:
!python -m pytest -q
!python scripts/evaluar_nlu.py

## 5. Cédulas de prueba para cada persona del equipo

Si dos personas usan la misma cédula, la cita que agenda una le aparece a la otra. Aquí cada quien recibe su juego de pacientes: uno para agendar desde cero, uno que ya tiene cita (para consultar, cambiar y cancelar) y uno con orden de especialista.

In [ ]:
PERSONAS = 5
!python scripts/cedulas_prueba.py --personas {PERSONAS}

## 6. Conversación completa de ejemplo

Usa a doña Rosa (cédula de demostración). Al final cancela la cita para que la celda se pueda repetir.

In [ ]:
!python -m magnolia.canales.consola --guion \
  "buenas, necesito una cita con el de la presión" \
  "treinta y dos millones cuatrocientos cincuenta y seis mil setecientos ochenta y nueve" \
  "sí señora" "ninguna, mejor la otra semana por la mañana" "¿cómo? repítame" "la segunda" "sí" \
  "me duele mucho el pecho" "quiero cancelar mi cita" "sí"

## 7. Conversar aquí mismo (texto, con respuesta en audio)

Escriba como lo haría un adulto mayor: *"buenas mija, necesito una cita pa' la muela"*, *"¿cuándo es mi cita?"*, *"cancélemela"*, *"mejor el martes por la tarde"*. Escriba **salir** para terminar o **reiniciar** para empezar con otra cédula.

In [ ]:
from magnolia.canales.cuaderno import chatear
chatear(voz=True)

## 8. Encender el bot de Telegram (notas de voz)

**Solo una persona del equipo** ejecuta esta celda; las demás solo le escriben o le mandan notas de voz al bot desde su celular.

1. Configure `TELEGRAM_TOKEN` (y ojalá `LLM_API_KEY`) en el paso 2 y vuelva a ejecutarlo.
2. Ejecute esta celda. El bot responde **mientras la celda esté corriendo**; para apagarlo, toque ■.
3. Cada persona busca el bot en Telegram por el usuario que le puso en @BotFather, toca *Iniciar* y usa **sus propias cédulas** del paso 5.

Colab apaga la sesión tras unos 90 minutos sin actividad en la pestaña (o 12 horas como máximo). Si se apaga, ejecute de nuevo los pasos 1, 2, 3 y 8.

In [ ]:
!python -m magnolia.canales.telegram_bot

## 9. Resultados de la prueba

Calcula las métricas (tasa de éxito al agendar, turnos y minutos hasta agendar, % de turnos no entendidos, % por voz…) y descarga la bitácora en CSV para revisarla en Excel o Google Sheets. Las columnas `intencion_correcta`, `entendio_bien` y `comentario` quedan vacías para que el equipo las llene; las frases mal entendidas sirven para mejorar el bot.

Ejecútela **antes** de cerrar Colab: al cerrarse la sesión, los archivos se borran.

In [ ]:
!python scripts/metricas.py
!python scripts/exportar_bitacora.py --salida bitacora_pruebas.csv
try:
    from google.colab import files
    files.download("bitacora_pruebas.csv")
except ImportError:
    print("Archivo guardado en bitacora_pruebas.csv")

---
**Sin clave de Groq pero con notas de voz:** antes del paso 8 ejecute `!pip install -q faster-whisper` y agregue la línea `STT_MODO=local` al archivo `.env` (más lento, corre en Colab).